# EDA Notebook 2 — Quiz Attempts Bronze
**Source**: `mini_project_grp6.bronze.quiz_attempts_bronze`  
**Format**: CSV | ~500K rows/day | Score, pass/fail, attempt_number 1–3  
**Goal**: Score distributions, pass rates, logical inconsistencies, attempt patterns

In [0]:
from pyspark.sql import functions as F

CATALOG = "mini_project_grp6"
TABLE   = f"{CATALOG}.bronze.quiz_attempts_bronze"

df      = spark.table(TABLE)
total   = df.count()

print(f"Total rows: {total:,}")
df.printSchema()

In [0]:
# ============================================================
# CELL 2 — Null Audit
# ============================================================

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()
null_counts.columns = ["column", "null_count"]
null_counts["null_pct"] = (null_counts["null_count"] / total * 100).round(2)
print(null_counts.sort_values("null_count", ascending=False).to_string(index=False))

In [0]:
# ============================================================
# CELL 3 — Status Distribution
# ============================================================
# PASSED | FAILED | IN_PROGRESS | TIMED_OUT

print("=== STATUS DISTRIBUTION ===")
display(
    df.groupBy("status")
      .agg(
          F.count("*").alias("count"),
          F.round(F.count("*") / total * 100, 2).alias("pct")
      )
      .orderBy("count", ascending=False)
)

In [0]:
# ============================================================
# CELL 4 — Score Validation (DQ Rule)
# ============================================================
# score_obtained must be between 0 and max_score

invalid_scores = df.filter(
    (F.col("score_obtained") < 0) |
    (F.col("score_obtained") > F.col("max_score"))
)

print(f"Invalid score records : {invalid_scores.count():,}")
if invalid_scores.count() > 0:
    display(invalid_scores.select(
        "attempt_id", "student_id", "score_obtained", "max_score"
    ).limit(20))

In [0]:
# ============================================================
# CELL 5 — IN_PROGRESS with submit_ts (Logical Inconsistency)
# ============================================================
# Business rule: IN_PROGRESS + non-null submit_ts = flag

logical_error = df.filter(
    (F.col("status") == "IN_PROGRESS") &
    F.col("submit_ts").isNotNull()
)

print(f"IN_PROGRESS with submit_ts (logical error) : {logical_error.count():,}")
display(logical_error.select(
    "attempt_id", "student_id", "status", "start_ts", "submit_ts"
).limit(10))

In [0]:
# ============================================================
# CELL 6 — Score & Pass Rate Distribution
# ============================================================

print("=== SCORE STATS ===")
df.select("score_obtained", "max_score", "pass_threshold").describe().show()

print("\n=== SCORE % PERCENTILES ===")
df.withColumn(
    "score_pct", F.col("score_obtained") / F.col("max_score") * 100
).selectExpr(
    "percentile_approx(score_pct, 0.10) as p10",
    "percentile_approx(score_pct, 0.25) as p25",
    "percentile_approx(score_pct, 0.50) as p50",
    "percentile_approx(score_pct, 0.75) as p75",
    "percentile_approx(score_pct, 0.90) as p90"
).show()

In [0]:
# ============================================================
# CELL 7 — Attempt Number Distribution
# ============================================================
# attempt_number should be 1, 2, or 3

display(
    df.groupBy("attempt_number")
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("score_obtained"), 2).alias("avg_score")
      )
      .orderBy("attempt_number")
)

# Flag any attempt_number outside 1-3
out_of_range = df.filter(
    (F.col("attempt_number") < 1) | (F.col("attempt_number") > 3)
)
print(f"\nAttempt number out of range [1,3]: {out_of_range.count():,}")

In [0]:
# ============================================================
# CELL 8 — Pass Rate by Attempt Number
# ============================================================

display(
    df.groupBy("attempt_number")
      .agg(
          F.round(
              F.sum((F.col("status") == "PASSED").cast("int")) /
              F.count("*") * 100, 2
          ).alias("pass_rate_pct")
      )
      .orderBy("attempt_number")
)

In [0]:
# ============================================================
# CELL 9 — Time Duration of Attempts
# ============================================================
# submit_ts - start_ts = time spent on quiz

df_with_duration = df.withColumn(
    "duration_mins",
    (F.unix_timestamp("submit_ts") - F.unix_timestamp("start_ts")) / 60
).filter(F.col("duration_mins").isNotNull())

print("Quiz duration (minutes):")
df_with_duration.selectExpr(
    "percentile_approx(duration_mins, 0.25) as p25",
    "percentile_approx(duration_mins, 0.50) as p50",
    "percentile_approx(duration_mins, 0.75) as p75",
    "percentile_approx(duration_mins, 0.99) as p99",
    "max(duration_mins) as max_mins"
).show()

# Flag negative durations
neg_duration = df_with_duration.filter(F.col("duration_mins") < 0)
print(f"Negative duration records: {neg_duration.count():,}")

In [0]:
# ============================================================
# CELL 10 — Course Coverage in Quiz Data
# ============================================================

print(f"Unique students in quiz data : {df.select('student_id').distinct().count():,}")
print(f"Unique courses in quiz data  : {df.select('course_id').distinct().count():,}")
print(f"Unique quizzes               : {df.select('quiz_id').distinct().count():,}")